# SIR Epidemiological Model
### A deterministic compartmental model with dissmodel

The SIR model tracks the spread of an infectious disease through a population
divided into three compartments:

- **S**usceptible — healthy individuals who can be infected
- **I**nfected — individuals who carry and spread the disease
- **R**ecovered — individuals who have recovered and gained immunity

## How it works

At each time step, the model computes:

| Variable | Formula | Description |
|:---|:---|:---|
| α | `contacts × probability` | Force of infection |
| ΔI | `I × α × (S / N)` | New infections |
| ΔR | `I / duration` | New recoveries |

Where **N = S + I + R** is the total population.

## Imports

In [4]:
from dissmodel.core import Environment
from dissmodel.visualization import Chart

In [6]:

from dissmodel.core import Model
from dissmodel.visualization import track_plot


@track_plot("Susceptible", "green")
@track_plot("Infected", "red")
@track_plot("Recovered", "blue")
class SIR(Model):
    """
    Deterministic SIR epidemiological model.

    Tracks the spread of an infectious disease through three compartments:
    susceptible, infected, and recovered. At each step, new infections and
    recoveries are computed based on contact rate, transmission probability,
    and disease duration.

    Parameters
    ----------
    susceptible : int, optional
        Initial number of susceptible individuals, by default 9998.
    infected : int, optional
        Initial number of infected individuals, by default 2.
    recovered : int, optional
        Initial number of recovered individuals, by default 0.
    duration : int, optional
        Average number of steps an individual remains infectious,
        by default 2.
    contacts : int, optional
        Average number of contacts per individual per step, by default 6.
    probability : float, optional
        Probability of transmission per contact with an infected individual,
        by default 0.25.

    Notes
    -----
    The ``@track_plot`` decorators register ``susceptible``, ``infected``,
    and ``recovered`` for automatic live plotting. Any
    :class:`~dissmodel.visualization.Chart` connected to the same environment
    will plot these variables at every step without any extra configuration.

    The force of infection at each step is computed as:

    .. math::

        \\alpha = contacts \\times probability

        \\Delta I = I \\times \\alpha \\times \\frac{S}{N}

        \\Delta R = \\frac{I}{duration}

    Examples
    --------
    >>> from dissmodel.core import Environment
    >>> env = Environment(end_time=30)
    >>> sir = SIR()
    >>> env.run()
    """

    #: Number of susceptible individuals.
    susceptible: float

    #: Number of infected individuals.
    infected: float

    #: Number of recovered individuals.
    recovered: float

    #: Average number of steps an individual remains infectious.
    duration: int

    #: Average number of contacts per individual per step.
    contacts: int

    #: Probability of transmission per contact.
    probability: float

    def setup(
        self,
        susceptible: int = 9998,
        infected: int = 2,
        recovered: int = 0,
        duration: int = 2,
        contacts: int = 6,
        probability: float = 0.25,
    ) -> None:
        """
        Configure the model parameters.

        Parameters
        ----------
        susceptible : int, optional
            Initial number of susceptible individuals, by default 9998.
        infected : int, optional
            Initial number of infected individuals, by default 2.
        recovered : int, optional
            Initial number of recovered individuals, by default 0.
        duration : int, optional
            Average number of steps an individual remains infectious,
            by default 2.
        contacts : int, optional
            Average number of contacts per individual per step,
            by default 6.
        probability : float, optional
            Probability of transmission per contact, by default 0.25.
        """
        self.susceptible = susceptible
        self.infected    = infected
        self.recovered   = recovered
        self.duration    = duration
        self.contacts    = contacts
        self.probability = probability

    def execute(self) -> None:
        """
        Advance the model by one time step.

        Computes new infections and recoveries, then updates the
        susceptible, infected, and recovered compartments.
        """
        total = self.susceptible + self.infected + self.recovered
        alpha = self.contacts * self.probability

        new_infected  = self.infected * alpha * (self.susceptible / total)
        new_recovered = self.infected / self.duration

        self.susceptible -= new_infected
        self.infected    += new_infected - new_recovered
        self.recovered   += new_recovered


__all__ = ["SIR"]


## ⚠️ Important: instantiation order

The `Environment` must always be created **before** any model (`SIR`, `Chart`).
Models connect to the active environment automatically when instantiated —
if no environment exists yet, the connection fails.

```
Environment  →  SIR  →  Chart
     ↑           ↑        ↑
  first       second    third
```

## Setting up the model

The SIR model accepts the following parameters:

| Parameter | Description | Default |
|:---|:---|:---|
| `susceptible` | Initial susceptible population | 9998 |
| `infected` | Initial infected population | 2 |
| `recovered` | Initial recovered population | 0 |
| `duration` | Steps an individual remains infectious | 2 |
| `contacts` | Average contacts per individual per step | 6 |
| `probability` | Transmission probability per contact | 0.25 |

In [10]:
# 1. Environment must be created first
env = Environment()

# 2. Model connects to the active environment automatically
SIR(
    susceptible=9998,
    infected=2,
    recovered=0,
    duration=2,
    contacts=6,
    probability=0.25,
)

# 3. Chart also connects to the active environment automatically
Chart(show_legend=True, show_grid=True, title="SIR Model")

env.run(30)

Output()

Running from 0 to 30 (duration: 30)


## Visualization

The `Chart` model redraws the time series at every simulation step.
It automatically tracks `susceptible`, `infected`, and `recovered`
because the `SIR` class is decorated with `@track_plot` — no extra
configuration needed.

Output()

Chart (chart.0)

## Try it yourself

Go back and experiment:

- Increase `contacts` to simulate a more contagious disease
- Increase `duration` to make the disease last longer
- Decrease `probability` to simulate a less transmissible disease
- Try starting with more `infected` individuals
- Change `env.run(30)` to run for more steps